In [1]:
#Notebook formatting
from IPython.display import display, HTML
display(HTML("<style>.jp-Cell { margin-left: -50% !important; margin-right: -50% !important; }</style>"))

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import gc
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pyarrow as pa
print('complete')

complete


In [3]:
sns.set_style("whitegrid")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
#pd.set_option('display.float_format', '{:,.2f}'.format)

In [4]:
%load_ext memory_profiler

In [5]:
%load_ext autotime

time: 48.5 μs (started: 2026-06-22 20:56:01 -05:00)


In [6]:
#Note - DFR load attempts that wont work:
#dfr_data = pq.read_table('LCR_cleaned_final.parquet', nrows=1_000_000)
#dfr = dfr_data.to_pandas().sample(100000, random_state=42)
#dfr = pd.read_parquet('LCR_cleaned_final.parquet')
#dfr_sample = dfr.sample(n=100000, random_state=42)

#dfr_data = pq.ParquetFile('LCR_cleaned_final.parquet')
#batch = next(pf.iter_batches(batch_size=1_000_000))
#dfr = pa.Table.from_batches([batch]).to_pandas().sample(100000, random_state=42)

#dataset = pq.ParquetDataset('LCR_cleaned_final')
#table = dataset.read_pandas(use_threads=True)
#dfr = table.to_pandas().head(1000000)

time: 539 μs (started: 2026-06-22 20:56:02 -05:00)


In [7]:
# This cell tests whether our dfr sample is representative of the full dataset.
# Run this cell first, compare the .describe() outputs, then comment out this cell and the cell underneath once a good sample size is confirmed.

# Reason: The full LCR_cleaned_final dataset is very large (~9GB, 27.6M rows). I used a smaller random sample for faster analysis and statistical representativeness.

risk_score_pop = pd.read_parquet('LCR_cleaned_final', columns=['risk_score', 'dti'])
risk_score_sample = risk_score_pop.sample(500000, random_state=42)
print('population description')
print(risk_score_pop.describe())
print(f"Row Count: {len(risk_score_pop)}")
print(f"   risk_score_pop memory usage: {risk_score_pop.memory_usage(deep=True).sum() / (1024**2):.1f} MB")

population description
       risk_score           dti
count   9151111.0  2.764874e+07
mean    628.17209  1.433401e+00
std     89.936793  1.053916e+02
min           0.0 -1.000000e-02
25%         591.0  8.060000e-02
50%         637.0  1.998000e-01
75%         675.0  3.661000e-01
max         990.0  5.000003e+05
Row Count: 27648741
   risk_score_pop memory usage: 659.2 MB
time: 2.36 s (started: 2026-06-22 20:56:02 -05:00)


In [8]:
print('sample description')
print(risk_score_sample.describe())
del risk_score_pop
del risk_score_sample

sample description
       risk_score            dti
count    165426.0  500000.000000
mean   628.122206       1.600408
std     89.824052      73.763559
min           0.0      -0.010000
25%         591.0       0.080600
50%         637.0       0.200300
75%         675.0       0.366400
max         990.0   44088.000000
time: 47.1 ms (started: 2026-06-22 20:56:06 -05:00)


In [9]:
print("Loading important columns for quick testing.")
cols = [
    'addr_state',
'annual_inc',
'delinq_2yrs',
'dti',
'emp_length_lt1',
'emp_length',
'fico_range_high',
'fico_range_low',
'home_ownership',
'inq_last_6mths',
'int_rate',
'issue_d',
'last_fico_range_high',
'last_fico_range_low',
'loan_amnt',
'loan_status',
'pub_rec',
'purpose',
'revol_bal',
'revol_util',
'sub_grade',
'term',
'verification_status',
]
dfa_data = pd.read_parquet('LCA_cleaned_final', columns=cols).head(500000) 
dfa = pd.read_parquet('LCA_cleaned_final', columns=cols) #2,260,701
#dfa = dfa_data.sample(100000, random_state=42)


dfr_data = ds.dataset('LCR_cleaned_final').scanner().head(1000).to_pandas() #Currently at 1k for basic analysis and saving RAM. Set to 500k for a correctly representative sample. 
#dfr = dfr_data.sample(500000, random_state=42)


#test = pd.read_parquet('LCR_cleaned_final') #27,648,741 rows. Appx ~9GB
print("loaded")

Loading important columns for quick testing.
loaded
time: 1.59 s (started: 2026-06-22 20:56:16 -05:00)


### Quick exploration

In [10]:
print(f"Loaded dfa: {dfa_data.shape[0]:,} rows, {dfa_data.shape[1]} columns")
print("Initial memory usage:")
print(f"   dfa: {dfa_data.memory_usage(deep=True).sum() / (1024**3):.1f} GB")
print(f"   dfr: {dfr_data.memory_usage(deep=True).sum() / (1024**2):.1f} MB")

Loaded dfa: 500,000 rows, 23 columns
Initial memory usage:
   dfa: 0.2 GB
   dfr: 0.2 MB
time: 236 ms (started: 2026-06-22 20:56:43 -05:00)


In [11]:
dfa_data.head(5)

,addr_state,annual_inc,delinq_2yrs,dti,emp_length_lt1,emp_length,fico_range_high,fico_range_low,home_ownership,inq_last_6mths,int_rate,issue_d,last_fico_range_high,last_fico_range_low,loan_amnt,loan_status,pub_rec,purpose,revol_bal,revol_util,sub_grade,term,verification_status
__null_dask_index__,,,,,,,,,,,,,,,,,,,,,,,
0,PA,55000.0,0,5.91,0,10,679.0,675.0,MORTGAGE,1,13.99,2015-12-01,564,560,3600.0,Fully Paid,0,debt_consolidation,2765.0,29.700001,C4,36,Not Verified
1,SD,65000.0,1,16.059999,0,10,719.0,715.0,MORTGAGE,4,11.99,2015-12-01,699,695,24700.0,Fully Paid,0,small_business,21470.0,19.200001,C1,36,Not Verified
2,IL,63000.0,0,10.78,0,10,699.0,695.0,MORTGAGE,0,10.78,2015-12-01,704,700,20000.0,Fully Paid,0,home_improvement,7869.0,56.200001,B4,60,Not Verified
3,NJ,110000.0,0,17.059999,0,10,789.0,785.0,MORTGAGE,0,14.85,2015-12-01,679,675,35000.0,Current,0,debt_consolidation,7802.0,11.6,C5,60,Source Verified
4,PA,104433.0,1,25.370001,0,3,699.0,695.0,MORTGAGE,3,22.450001,2015-12-01,704,700,10400.0,Fully Paid,0,major_purchase,21929.0,64.5,F1,60,Source Verified


time: 7.52 ms (started: 2026-06-22 20:56:46 -05:00)


In [12]:
dfr_data.head(5)

,amount_requested,application_date,loan_title,risk_score,dti,zip_code,state,emp_length,policy_code,emp_length_lt1
__null_dask_index__,,,,,,,,,,
0,1000.0,2007-05-26,Wedding Covered but No Honeymoon,693.0,0.1000,481xx,NM,4,0.0,0
1,1000.0,2007-05-26,Consolidating Debt,703.0,0.1000,010xx,MA,1,0.0,1
2,11000.0,2007-05-27,Want to consolidate my debt,715.0,0.1000,212xx,MD,1,0.0,0
3,6000.0,2007-05-27,waksman,698.0,0.3864,017xx,MA,1,0.0,1
4,1500.0,2007-05-27,mdrigo,509.0,0.0943,209xx,MD,1,0.0,1


time: 7.28 ms (started: 2026-06-22 20:56:46 -05:00)


In [13]:
dfa_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 500000 entries, 0 to 57571
Data columns (total 23 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   addr_state            499996 non-null  string        
 1   annual_inc            500000 non-null  Float64       
 2   delinq_2yrs           500000 non-null  Int32         
 3   dti                   499907 non-null  Float64       
 4   emp_length_lt1        500000 non-null  Int32         
 5   emp_length            500000 non-null  Int32         
 6   fico_range_high       499996 non-null  Float64       
 7   fico_range_low        499996 non-null  Float64       
 8   home_ownership        499996 non-null  string        
 9   inq_last_6mths        500000 non-null  Int32         
 10  int_rate              499996 non-null  Float64       
 11  issue_d               499996 non-null  datetime64[ns]
 12  last_fico_range_high  499996 non-null  Int32         
 13  last_

In [ ]:
dfr_data.info()

In [ ]:
dfa_data.describe().round(2)

In [ ]:
dfr_data.describe().round(2)

### Creating cohort framework and more testing

In [28]:
status_cts = dfa_data['loan_status'].value_counts()
status_p = dfa_data['loan_status'].value_counts(normalize=True)
by_year_cts = dfa_data['issue_d'].dt.year.value_counts()
by_year_p = dfa_data['issue_d'].dt.year.value_counts(normalize=True)
by_purpose_cts = dfa_data['purpose'].value_counts()
by_purpose_p = dfa_data['purpose'].value_counts(normalize=True)
by_state_cts = dfa_data['addr_state'].value_counts()
by_state_p = dfa_data['addr_state'].value_counts(normalize=True)
by_grade_cts = dfa_data['sub_grade'].value_counts()
by_grade_p = dfa_data['sub_grade'].value_counts(normalize=True)

dlqcy = ['Charged Off','Late (31-120 days)','Late (16-30 days)','Default']
delinquencies = dfa_data['loan_status'].isin(dlqcy).value_counts()
delinquencies_p = dfa_data['loan_status'].isin(dlqcy).value_counts(normalize=True)
delinquencies_df = dfa_data[dfa_data['loan_status'].isin(dlqcy)]

dfa_data['delinquency_tf'] = dfa_data['loan_status'].isin(dlqcy).astype(int)
dflt = ['Charged Off','Default']
dfa_data['defaulted_tf'] = dfa_data['loan_status'].isin(dflt).astype(int)


dfa['defaulted_tf'] = dfa['loan_status'].isin(dflt).astype(int)
dfa['delinquency_tf'] = dfa['loan_status'].isin(dlqcy).astype(int)

time: 407 ms (started: 2026-06-22 21:02:27 -05:00)


In [15]:
delinquency_by_subgrade = dfa_data.groupby('sub_grade')['delinquency_tf'].value_counts()

time: 34.6 ms (started: 2026-06-22 20:57:02 -05:00)


In [16]:
#Creating delinquency rate/proportion cohorts by:
# Sub grade
# Issuance date
# Purpose
# State
# Employment length of one year or less.
# Employment Length
# Debt-to-Income ratio


delinquency_rate_subgrade = dfa_data.groupby('sub_grade')['delinquency_tf'].mean() 
delinquency_rate_year = dfa_data.groupby(dfa_data.issue_d.dt.year)['delinquency_tf'].mean() 
delinquency_rate_purpose = dfa_data.groupby('purpose')['delinquency_tf'].mean() 
delinquency_rate_state = dfa_data.groupby('addr_state')['delinquency_tf'].mean() 
delinquency_rate_low_employment = dfa_data.groupby('emp_length_lt1')['delinquency_tf'].mean() 
delinquency_rate_emp_length = dfa_data.groupby('emp_length')['delinquency_tf'].mean() 
delinquency_rate_dti = dfa_data.groupby(pd.cut(dfa_data['dti'], bins=10), observed=True)['delinquency_tf'].mean()

default_rate_subgrade = dfa_data.groupby('sub_grade')['defaulted_tf'].mean() 
default_rate_year = dfa_data.groupby(dfa_data.issue_d.dt.year)['defaulted_tf'].mean() 
default_rate_purpose = dfa_data.groupby('purpose')['defaulted_tf'].mean() 
default_rate_state = dfa_data.groupby('addr_state')['defaulted_tf'].mean() 
default_rate_low_employment = dfa_data.groupby('emp_length_lt1')['defaulted_tf'].mean() 
default_rate_emp_length = dfa_data.groupby('emp_length')['defaulted_tf'].mean() 
default_rate_dti = dfa_data.groupby(pd.cut(dfa_data['dti'], bins=10), observed=True)['defaulted_tf'].mean()

time: 17.2 s (started: 2026-06-22 20:57:12 -05:00)


In [17]:
delinquency_rate_year

issue_d
2014.0    0.180318
2015.0    0.180496
2016.0    0.179248
2017.0    0.121472
Name: delinquency_tf, dtype: float64

time: 1.81 ms (started: 2026-06-22 20:57:44 -05:00)


In [18]:
default_rate_year

issue_d
2014.0    0.178175
2015.0    0.175769
2016.0    0.170226
2017.0    0.101379
Name: defaulted_tf, dtype: float64

time: 1.83 ms (started: 2026-06-22 20:57:45 -05:00)


In [19]:
dfa_data.groupby(dfa_data.issue_d.dt.year)['delinquency_tf'].value_counts() 

issue_d  delinquency_tf
2014.0   0                  68068
         1                  14974
2015.0   0                 150120
         1                  33064
2016.0   0                 109888
         1                  23999
2017.0   0                  87750
         1                  12133
Name: count, dtype: int64

time: 6.62 s (started: 2026-06-22 20:57:46 -05:00)


In [20]:
year_summary = dfa_data.groupby(dfa_data['issue_d'].dt.year).agg(
    total_loans=('loan_status', 'count'),
    delinquencies=('delinquency_tf', 'sum'),
    defaults=('defaulted_tf', 'sum')
)

year_summary['delinquency_rate'] = year_summary['delinquencies'] / year_summary['total_loans']
year_summary['default_rate'] = year_summary['defaults'] / year_summary['total_loans']

year_summary['delinquency_rate'] = (year_summary['delinquency_rate'] * 100).round(2)
year_summary['default_rate'] = (year_summary['default_rate'] * 100).round(2)

print(year_summary)

         total_loans  delinquencies  defaults  delinquency_rate  default_rate
issue_d                                                                      
2014.0         83042          14974     14796             18.03         17.82
2015.0        183184          33064     32198             18.05         17.58
2016.0        133887          23999     22791             17.92         17.02
2017.0         99883          12133     10126             12.15         10.14
time: 6.62 s (started: 2026-06-22 20:58:11 -05:00)


In [21]:
year_summary['delinquencies'] - year_summary['defaults']

issue_d
2014.0     178
2015.0     866
2016.0    1208
2017.0    2007
dtype: int64

time: 2.03 ms (started: 2026-06-22 20:58:19 -05:00)


In [22]:
status_cts

loan_status
Fully Paid            312319
Current               102031
Charged Off            79904
Late (31-120 days)      3599
In Grace Period         1476
Late (16-30 days)        660
Default                    7
Name: count, dtype: Int64

time: 1.97 ms (started: 2026-06-22 20:58:20 -05:00)


In [23]:
loan_status_by_year = dfa_data.groupby([dfa_data['issue_d'].dt.year, 'loan_status']).size()

print(loan_status_by_year)

issue_d  loan_status       
2014.0   Charged Off            14796
         Current                 6002
         Fully Paid             61977
         In Grace Period           89
         Late (16-30 days)         29
         Late (31-120 days)       149
2015.0   Charged Off            32197
         Current                20809
         Default                    1
         Fully Paid            128981
         In Grace Period          330
         Late (16-30 days)        137
         Late (31-120 days)       729
2016.0   Charged Off            22790
         Current                20336
         Default                    1
         Fully Paid             89143
         In Grace Period          409
         Late (16-30 days)        176
         Late (31-120 days)      1032
2017.0   Charged Off            10121
         Current                54884
         Default                    5
         Fully Paid             32218
         In Grace Period          648
         Late (16-30 d

In [ ]:
vintage = dfa_data.groupby(dfa_data['issue_d'].dt.year).agg(
    total_loans=('loan_status', 'count'),
    charged_off=('loan_status', lambda x: (x == 'Charged Off').sum()),
    default=('loan_status', lambda x: (x == 'Default').sum()),
    late_31_120=('loan_status', lambda x: (x == 'Late (31-120 days)').sum()),
    fully_paid=('loan_status', lambda x: (x == 'Fully Paid').sum()),
    current=('loan_status', lambda x: (x == 'Current').sum())
)

vintage['delinquency_rate'] = (vintage['charged_off'] + vintage['late_31_120']) / vintage['total_loans'] * 100
vintage['default_rate'] = vintage['charged_off'] / vintage['total_loans'] * 100   # Most common definition

vintage = vintage.round(2)
print(vintage)

### Quick reflection:

- Lending Club experienced explosive growth in loan volume starting in 2013, peaking in 2017–2018.
- 2007–2012 had lower volume but relatively clean performance.
- From 2013–2016, both delinquency and charge-off rates increased significantly. 
- "Default" status is rare in the data. Most serious outcomes appear as "Charged Off".
- Late payments grew substantially after 2013. Could be due to increasing borrower stress after 2008 recession, or Lending Club's willingness to take on riskier loans. 

##### Some questions I'm considering:

- Why are outright "Defaults" rare, while "Charged Off" numbers are much higher? What does this pattern signal about their business practices, collections, and underwriting strategy?
- What does the steady rise in late payment statuses imply about borrower behavior and portfolio health?
- What incentives might Lending Club have had for allowing more loans to reach "Charged Off" status rather than earlier default?

In [24]:
from scipy.stats import chi2_contingency, pearsonr, chi2
categorical_columns = ['addr_state', 'home_ownership', 'purpose', 'sub_grade', 'verification_status']

a = 0.05

for col in categorical_columns:
    contingency_table = pd.crosstab(dfa_data[col], dfa_data['delinquency_tf'])
    chi_sq, pval, dof, exp = chi2_contingency(contingency_table)
    crit_val = chi2.ppf((1-a), dof)
    print(f"{col}:------- \nchi2 = {chi_sq:.2f} \ncritical_value = {crit_val:.2f} \nsignificance ratio = {chi_sq / crit_val:.2f} \np-value = {pval} \ndof = {dof} \n\n")

addr_state:------- 
chi2 = 1008.95 
critical_value = 66.34 
significance ratio = 15.21 
p-value = 2.2187727469354612e-179 
dof = 49 


home_ownership:------- 
chi2 = 2230.12 
critical_value = 9.49 
significance ratio = 235.05 
p-value = 0.0 
dof = 4 


purpose:------- 
chi2 = 1291.21 
critical_value = 21.03 
significance ratio = 61.41 
p-value = 3.8942960453306e-269 
dof = 12 


sub_grade:------- 
chi2 = 32628.85 
critical_value = 48.60 
significance ratio = 671.34 
p-value = 0.0 
dof = 34 


verification_status:------- 
chi2 = 4774.00 
critical_value = 5.99 
significance ratio = 796.80 
p-value = 0.0 
dof = 2 


time: 225 ms (started: 2026-06-22 20:59:09 -05:00)


### Chi-Square Test Results & Quick Observations

I ran Chi-square tests on several categorical variables to see how strongly they’re associated with `delinquency_tf`.

When I tested on the full dataset, almost every p-value came back as 0.0. That’s technically good news, but it doesn’t help me figure out which variables are actually the *strongest*. So I ran the tests on both the full population and different sample sizes to get a better sense of relative strength.

#### What is the Significance Ratio?

I’m still building my intuition around chi-square results. After some side research, I realized there isn’t one universal cutoff people use. So I started calculating a simple **Significance Ratio** to help me compare:

**Significance Ratio = chi² statistic ÷ critical value**

It should indicate **how many times stronger** the observed relationship is compared to what we’d expect if the variables were completely unrelated (pure chance).

**Example:**
- `sub_grade` had a chi² of 32,629 and a critical value around 66.34 → Significance Ratio ≈ **671x**

That’s an extremely strong signal.

#### Quick Observations:
- All the variables I tested are statistically significant.
- **`sub_grade`** is by far the strongest predictor — no surprise since it’s Lending Club’s own risk rating.
- I’m keeping an eye on overfitting risk, especially with high-cardinality variables like `addr_state` and `purpose`.

This step is helping me prioritize which categorical variables are actually worth keeping for the modeling phase.

In [31]:
numeric_cols = ['annual_inc', 'delinq_2yrs', 'dti', 'emp_length', 
                   'fico_range_high', 'fico_range_low', 'inq_last_6mths', 
                   'int_rate', 'loan_amnt', 'pub_rec', 'revol_bal', 
                   'revol_util', 'term']

s_corr_defaults = []
s_corr_delinquencies = []

for col in numeric_cols:
    s_1corr, p1_val = pearsonr(dfa_data[col], dfa_data['defaulted_tf'])
    s_corr_defaults.append({'variable': col,'correlation': s_1corr, 'p_value': p1_val})
s_corr_dflt = pd.DataFrame(s_corr_defaults)
s_corr_dflt = s_corr_dflt.sort_values('correlation', ascending=False)

for col in numeric_cols:
    s_2corr, p2_val = pearsonr(dfa_data[col], dfa_data['delinquency_tf'])
    s_corr_delinquencies.append({'variable': col,'correlation': s_2corr, 'p_value': p2_val})
s_corr_dlqcy = pd.DataFrame(s_corr_delinquencies)
s_corr_dlqcy = s_corr_dlqcy.sort_values('correlation', ascending=False)

print(f"DEFAULTS:\n{s_corr_dflt.round(4)}\n\n")
print(f"DELINQUENCY:\n{s_corr_dlqcy.round(4)}\n\n")




p_corr_defaults = []
p_corr_delinquencies = []

for col in numeric_cols:
    p_1corr, p3_val = pearsonr(dfa[col], dfa['defaulted_tf'])
    p_corr_defaults.append({'variable': col,'correlation': p_1corr, 'p_value': p3_val})
p_corr_dflt = pd.DataFrame(p_corr_defaults)
p_corr_dflt = p_corr_dflt.sort_values('correlation', ascending=False)

for col in numeric_cols:
    p_2corr, p4_val = pearsonr(dfa[col], dfa['delinquency_tf'])
    p_corr_delinquencies.append({'variable': col,'correlation': p_2corr, 'p_value': p4_val})
p_corr_dlqcy = pd.DataFrame(p_corr_delinquencies)
p_corr_dlqcy = p_corr_dlqcy.sort_values('correlation', ascending=False)


print(f"POPULATION DEFAULTS:\n{p_corr_dflt.round(4)}\n\n")
print(f"POPULATION DELINQUENCY:\n{p_corr_dlqcy.round(4)}\n\n")

DEFAULTS:
           variable  correlation  p_value
6    inq_last_6mths       0.0872      0.0
8         loan_amnt       0.0277      0.0
9           pub_rec       0.0269      0.0
1       delinq_2yrs       0.0107      0.0
3        emp_length      -0.0284      0.0
10        revol_bal      -0.0306      0.0
0        annual_inc      -0.0424      0.0
2               dti          NaN      NaN
4   fico_range_high          NaN      NaN
5    fico_range_low          NaN      NaN
7          int_rate          NaN      NaN
11       revol_util          NaN      NaN
12             term          NaN      NaN


DELINQUENCY:
           variable  correlation  p_value
6    inq_last_6mths       0.0873      0.0
8         loan_amnt       0.0327      0.0
9           pub_rec       0.0279      0.0
1       delinq_2yrs       0.0129      0.0
3        emp_length      -0.0285      0.0
10        revol_bal      -0.0308      0.0
0        annual_inc      -0.0414      0.0
2               dti          NaN      NaN
4   fico_

In [ ]:
plt.figure(figsize=(10, 6))
dfa['loan_status'].value_counts().plot(kind='bar')

plt.title('Loan Status Distribution')
plt.xlabel('Loan Status')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
#The best way is to store the results in a dictionary or DataFrame while running the loop, so you can pull specific values later.

In [ ]:
from scipy.stats import chi2_contingency, chi2

categorical_columns = ['addr_state', 'home_ownership', 'purpose', 'sub_grade', 'verification_status']
alpha = 0.05

pval = {}   # ← Add this line

for col in categorical_columns:
    contingency_table = pd.crosstab(dfa_data[col], dfa_data['delinquency_tf'])
    chi_sq, p_value, dof, exp = chi2_contingency(contingency_table)
    crit_val = chi2.ppf(1 - alpha, dof)
    
    pval[col] = p_value   # ← Save the p-value
    
    print(f"{col}: chi2 = {chi_sq:.2f} | p = {p_value:.2e} | ratio = {chi_sq / crit_val:.2f}x")

# Now you can easily access any p-value:
print("\nSpecific p-values:")
print(f"addr_state p-value: {pval['addr_state']}")
print(f"sub_grade p-value:   {pval['sub_grade']}")